In [1]:
!pip install av tqdm numpy pandas scikit-learn matplotlib numpy transformers

In [2]:
# ============================================================================
# CONFIGURATION & IMPORTS
# ============================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# ============================================================================
# HYPERPARAMETER CONFIGURATION
# NOTE: Dimensions chosen for GPU efficiency (divisible by 8/16/32)
# ============================================================================

CONFIG = {
    # -------------------------------------------------------------------------
    # MODEL SELECTION
    # -------------------------------------------------------------------------
    'vjepa_model': 'facebook/vjepa2-vitl-fpc64-256',
    'clip_model': 'openai/clip-vit-large-patch14',
    
    # -------------------------------------------------------------------------
    # WINDOWED FRAME SAMPLING (NEW in v2!)
    # -------------------------------------------------------------------------
    'num_frames': 64,           # Divisible by 16 (V-JEPA temporal patches)
    'window_stride': 32,        # 50% overlap
    'max_windows_per_video': 10,
    'min_video_frames': 64,
    'clip_sample_frames': 8,    # Divisible by 8
    
    # -------------------------------------------------------------------------
    # DATA
    # -------------------------------------------------------------------------
    'video_dir': './videos',
    'num_videos': None,
    'test_split': 0.15,
    'random_seed': 42,
    
    # -------------------------------------------------------------------------
    # TRANSLATOR NETWORK
    # -------------------------------------------------------------------------
    'translator_hidden_dim': 512,  # Divisible by 128
    'translator_num_blocks': 4,
    'translator_dropout': 0.1,
    
    # -------------------------------------------------------------------------
    # TRAINING
    # -------------------------------------------------------------------------
    'epochs': 500,
    'batch_size': 32,              # Divisible by 8
    'accumulation_steps': 1,       # Increase for larger effective batch
    'lr': 3e-4,
    'weight_decay': 0.05,
    'warmup_epochs': 30,
    'patience': 50,
    'temperature': 0.07,
    'contrastive_weight': 0.7,
    'cosine_weight': 0.3,
    'grad_clip': 1.0,
    
    # -------------------------------------------------------------------------
    # MEMORY OPTIMIZATION
    # -------------------------------------------------------------------------
    'use_amp': True,               # Automatic mixed precision
    'precompute_embeddings': True, # Pre-extract embeddings (saves GPU during training)
    'offload_to_cpu': True,        # Store embeddings on CPU
}

# ============================================================================
# DEVICE SETUP WITH MEMORY-AWARE CONFIGURATION
# ============================================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name()}")
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Memory: {total_mem:.1f} GB")
    
    # Adjust for low memory
    if total_mem < 12:
        CONFIG['batch_size'] = 16
        CONFIG['accumulation_steps'] = 2
        print(f"  → Low memory mode: batch={CONFIG['batch_size']}, accum={CONFIG['accumulation_steps']}")
    
    # Enable TF32 for Ampere+ GPUs
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

# Memory management utilities
def clear_gpu_memory():
    """Clear GPU cache to free memory"""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

def get_gpu_memory_mb():
    """Get current GPU memory usage in MB"""
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / 1e6
    return 0

print(f"\n{'='*60}")
print("CONFIGURATION")
print('='*60)
print(f"Window: {CONFIG['num_frames']} frames, stride={CONFIG['window_stride']}")
print(f"Training: batch={CONFIG['batch_size']} × {CONFIG['accumulation_steps']} = {CONFIG['batch_size'] * CONFIG['accumulation_steps']} effective")
print(f"AMP: {CONFIG['use_amp']}, Pre-compute: {CONFIG['precompute_embeddings']}")
print('='*60)

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
GPU: NVIDIA A40
Memory: 47.7 GB

CONFIGURATION
Window: 64 frames, stride=32
Training: batch=32 × 1 = 32 effective
AMP: True, Pre-compute: True


In [3]:
# =============================================================================
# CONFIGURATION
# =============================================================================
# All hyperparameters centralized here for easy experimentation
# =============================================================================

CONFIG = {
    # Model identifiers
    "vjepa_model": "facebook/vjepa2-vitl-fpc64-256",  # V-JEPA2 ViT-Large, 64 frames per clip
    "clip_model": "openai/clip-vit-large-patch14",    # CLIP ViT-Large for target embeddings
    
    # Video sampling parameters
    "num_frames": 64,           # V-JEPA expects exactly 64 consecutive frames
    "window_stride": 32,        # Stride between windows (32 = 50% overlap)
    "max_windows_per_video": 5, # Cap windows per video to balance dataset
    "frame_size": 256,          # Frame resize dimension
    
    # Training hyperparameters  
    "batch_size": 4,            # Batch size (limited by GPU memory)
    "learning_rate": 1e-4,      # AdamW learning rate
    "num_epochs": 20,           # Training epochs
    "weight_decay": 0.01,       # L2 regularization
    "temperature": 0.07,        # InfoNCE temperature (CLIP default)
    
    # Translator architecture
    "vjepa_dim": 1024,          # V-JEPA ViT-L hidden dimension
    "clip_dim": 768,            # CLIP ViT-L embedding dimension
    "hidden_dim": 1024,         # Translator hidden layer size
    "dropout": 0.1,             # Dropout for regularization
    
    # Data paths
    "video_dir": "./videos",    # Directory containing video files
    "train_split": 0.8,         # Fraction for training (rest is validation)
    
    # Device
    "device": "cuda",           # Use GPU if available
}

print("Configuration loaded:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

Configuration loaded:
  vjepa_model: facebook/vjepa2-vitl-fpc64-256
  clip_model: openai/clip-vit-large-patch14
  num_frames: 64
  window_stride: 32
  max_windows_per_video: 5
  frame_size: 256
  batch_size: 4
  learning_rate: 0.0001
  num_epochs: 20
  weight_decay: 0.01
  temperature: 0.07
  vjepa_dim: 1024
  clip_dim: 768
  hidden_dim: 1024
  dropout: 0.1
  video_dir: ./videos
  train_split: 0.8
  device: cuda


In [4]:
# =============================================================================
# IMPORTS
# =============================================================================
import os
import random
import subprocess
import urllib.request
from pathlib import Path
from typing import List, Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import av
import numpy as np
from PIL import Image

from transformers import AutoProcessor, AutoModel, CLIPProcessor, CLIPModel
from tqdm.auto import tqdm

torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

if CONFIG["device"] == "cuda" and not torch.cuda.is_available():
    CONFIG["device"] = "cpu"
    print("CUDA not available, using CPU")
else:
    print(f"Using device: {CONFIG['device']}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: NVIDIA A40


In [5]:
# =============================================================================
# VIDEO DOWNLOAD UTILITIES
# =============================================================================

def download_with_ytdlp(url: str, output_path: str, max_duration: int = 60) -> bool:
    """Download video using yt-dlp."""
    try:
        cmd = [
            "yt-dlp",
            "--format", "bestvideo[height<=480][ext=mp4]+bestaudio[ext=m4a]/best[height<=480][ext=mp4]/best",
            "--match-filter", f"duration<={max_duration}",
            "--output", output_path,
            "--no-playlist", "--quiet",
            url
        ]
        result = subprocess.run(cmd, capture_output=True, timeout=120)
        return result.returncode == 0 and os.path.exists(output_path)
    except Exception as e:
        print(f"yt-dlp error: {e}")
        return False

def download_from_archive(identifier: str, output_path: str) -> bool:
    """Download video from Internet Archive."""
    try:
        for ext in [".mp4", ".ogv", ".avi"]:
            url = f"https://archive.org/download/{identifier}/{identifier}{ext}"
            try:
                urllib.request.urlretrieve(url, output_path)
                if os.path.exists(output_path) and os.path.getsize(output_path) > 1000:
                    return True
            except:
                continue
        return False
    except Exception as e:
        print(f"Archive download error: {e}")
        return False

video_dir = Path(CONFIG["video_dir"])
if video_dir.exists():
    existing_videos = list(video_dir.glob("*.mp4"))
    print(f"Found {len(existing_videos)} existing videos in {video_dir}")
else:
    print(f"Video directory {video_dir} does not exist.")

Found 253 existing videos in videos


In [6]:
# =============================================================================
# WINDOWED FRAME EXTRACTION
# =============================================================================

def extract_frames_from_video(video_path: str) -> Optional[np.ndarray]:
    """Extract all frames from a video file."""
    try:
        container = av.open(video_path)
        frames = [frame.to_ndarray(format="rgb24") for frame in container.decode(video=0)]
        container.close()
        return np.stack(frames) if frames else None
    except Exception as e:
        print(f"Error: {e}")
        return None

def get_consecutive_windows(frames: np.ndarray, window_size: int = 64, stride: int = 32, max_windows: int = 5) -> List[np.ndarray]:
    """Extract consecutive frame windows from video."""
    if len(frames) < window_size:
        return []
    windows = []
    start_idx = 0
    while start_idx + window_size <= len(frames) and len(windows) < max_windows:
        windows.append(frames[start_idx:start_idx + window_size])
        start_idx += stride
    return windows

def get_middle_frame(window: np.ndarray) -> Image.Image:
    """Get middle frame as PIL Image."""
    return Image.fromarray(window[len(window) // 2])


In [7]:
# =============================================================================
# LOAD V-JEPA AND CLIP MODELS (Memory Optimized)
# =============================================================================

from transformers import AutoVideoProcessor, AutoModel, CLIPProcessor, CLIPModel

print("Loading V-JEPA...")
vjepa_processor = AutoVideoProcessor.from_pretrained(CONFIG["vjepa_model"])
vjepa_model = AutoModel.from_pretrained(CONFIG["vjepa_model"]).to(device).eval()
for param in vjepa_model.parameters():
    param.requires_grad = False
print(f"V-JEPA: {sum(p.numel() for p in vjepa_model.parameters()) / 1e6:.1f}M params")

print("Loading CLIP (float16 for memory efficiency)...")
clip_processor = CLIPProcessor.from_pretrained(CONFIG["clip_model"])
clip_model = CLIPModel.from_pretrained(CONFIG["clip_model"], torch_dtype=torch.float16).to(device).eval()
for param in clip_model.parameters():
    param.requires_grad = False
print(f"CLIP: {sum(p.numel() for p in clip_model.parameters()) / 1e6:.1f}M params")

clear_gpu_memory()
print(f"GPU memory after model loading: {get_gpu_memory_mb():.0f} MB")

Loading V-JEPA...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


V-JEPA: 326.0M params
Loading CLIP (float16 for memory efficiency)...


`torch_dtype` is deprecated! Use `dtype` instead!


CLIP: 427.6M params
GPU memory after model loading: 2170 MB


In [8]:
# =============================================================================
# TRANSLATOR NETWORK
# =============================================================================

# Get model dimensions
vjepa_dim = vjepa_model.config.hidden_size  # 1024 for ViT-L
clip_dim = clip_model.config.projection_dim  # 768 for CLIP ViT-L

class TranslatorMLP(nn.Module):
    """MLP to translate V-JEPA embeddings to CLIP space."""
    
    def __init__(self, input_dim=1024, hidden_dim=512, output_dim=768, dropout=0.1):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim)
        )
        nn.init.xavier_uniform_(self.network[-1].weight, gain=0.1)
        nn.init.zeros_(self.network[-1].bias)
    
    def forward(self, x):
        return F.normalize(self.network(x), dim=-1)

translator = TranslatorMLP(
    input_dim=vjepa_dim, 
    hidden_dim=CONFIG['translator_hidden_dim'], 
    output_dim=clip_dim, 
    dropout=CONFIG['translator_dropout']
).to(device)
print(f"Translator: {sum(p.numel() for p in translator.parameters()) / 1e6:.2f}M params")
print(f"  Input: {vjepa_dim} (V-JEPA) → Output: {clip_dim} (CLIP)")

KeyError: 'translator_hidden_dim'

In [ ]:
# =============================================================================
# DATASET (Memory Optimized)
# =============================================================================
# Instead of storing frame arrays in memory (huge!), we:
# 1. Build an index of (video_path, start_frame) tuples
# 2. Pre-compute embeddings and store on CPU
# 3. Training only uses embeddings (no frame loading during training)
# =============================================================================

def get_video_frame_count(video_path):
    """Get total frames in video without loading all frames."""
    try:
        container = av.open(str(video_path))
        stream = container.streams.video[0]
        count = stream.frames or sum(1 for _ in container.decode(video=0))
        container.close()
        return count
    except:
        return 0

def extract_window(video_path, start_frame, num_frames):
    """Extract a specific window of frames from video."""
    container = av.open(str(video_path))
    frames = []
    for i, frame in enumerate(container.decode(video=0)):
        if i >= start_frame:
            frames.append(frame.to_ndarray(format='rgb24'))
        if len(frames) >= num_frames:
            break
    container.close()
    
    while len(frames) < num_frames:
        frames.append(frames[-1] if frames else np.zeros((224, 224, 3), dtype=np.uint8))
    return frames[:num_frames]

def compute_vjepa_embedding(frames):
    """Compute V-JEPA embedding for a window. Returns CPU tensor."""
    video_array = np.stack(frames, axis=0)
    inputs = vjepa_processor(video_array, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad(), torch.cuda.amp.autocast(enabled=CONFIG['use_amp']):
        outputs = vjepa_model(**inputs)
        hidden = outputs.last_hidden_state.squeeze(0)
        emb = hidden.mean(dim=0)
    return emb.float().cpu()

def compute_clip_embedding(frames, sample_frames=8):
    """Compute CLIP embedding for a window. Returns CPU tensor."""
    indices = np.linspace(0, len(frames)-1, sample_frames, dtype=int)
    sampled = [frames[i] for i in indices]
    
    embeddings = []
    for frame in sampled:
        img = Image.fromarray(frame)
        inputs = clip_processor(images=img, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad(), torch.cuda.amp.autocast(enabled=CONFIG['use_amp']):
            emb = clip_model.get_image_features(**inputs)
        embeddings.append(emb.squeeze().float().cpu())
    
    avg_emb = torch.stack(embeddings).mean(dim=0)
    return F.normalize(avg_emb, dim=0)

# Build window index (memory efficient - only stores paths/indices)
print("Building window index...")
window_index = []  # (video_path, start_frame)

for vpath in tqdm(video_dir.glob("*.mp4"), desc="Scanning videos"):
    total_frames = get_video_frame_count(str(vpath))
    if total_frames < CONFIG['min_video_frames']:
        continue
    
    # Calculate window starts
    starts = []
    pos = 0
    while pos + CONFIG['num_frames'] <= total_frames:
        starts.append(pos)
        pos += CONFIG['window_stride']
    
    # Limit windows per video
    if len(starts) > CONFIG['max_windows_per_video']:
        indices = np.linspace(0, len(starts)-1, CONFIG['max_windows_per_video'], dtype=int)
        starts = [starts[i] for i in indices]
    
    for start in starts:
        window_index.append((str(vpath), start))

print(f"Window index: {len(window_index)} windows")

# Pre-compute embeddings (stored on CPU)
print("\nPre-computing embeddings (stored on CPU)...")
vjepa_embeddings = []
clip_embeddings = []

for i, (vpath, start) in enumerate(tqdm(window_index, desc="Extracting embeddings")):
    frames = extract_window(vpath, start, CONFIG['num_frames'])
    
    vjepa_emb = compute_vjepa_embedding(frames)
    clip_emb = compute_clip_embedding(frames, CONFIG['clip_sample_frames'])
    
    vjepa_embeddings.append(vjepa_emb)
    clip_embeddings.append(clip_emb)
    
    if (i + 1) % 100 == 0:
        clear_gpu_memory()

vjepa_embeddings = torch.stack(vjepa_embeddings)  # [N, vjepa_dim] on CPU
clip_embeddings = torch.stack(clip_embeddings)    # [N, clip_dim] on CPU
clear_gpu_memory()

print(f"\n✅ Pre-computed embeddings:")
print(f"   V-JEPA: {vjepa_embeddings.shape}")
print(f"   CLIP:   {clip_embeddings.shape}")
print(f"   Storage: CPU (moved to GPU per-batch during training)")

# Train/test split
n_samples = len(window_index)
indices = np.arange(n_samples)
train_idx, test_idx = train_test_split(indices, test_size=CONFIG['test_split'], random_state=CONFIG['random_seed'])

X_train = vjepa_embeddings[train_idx]  # CPU
Y_train = clip_embeddings[train_idx]   # CPU
X_test = vjepa_embeddings[test_idx]    # CPU
Y_test = clip_embeddings[test_idx]     # CPU

print(f"\nDataset split:")
print(f"  Train: {len(train_idx)} windows")
print(f"  Test:  {len(test_idx)} windows")

In [ ]:
# =============================================================================
# DATALOADERS (Memory Optimized)
# =============================================================================
# Since embeddings are pre-computed and stored on CPU, we just need
# simple index-based batching. Data is moved to GPU per-batch.
# =============================================================================

from torch.utils.data import TensorDataset, DataLoader

# Create simple tensor datasets for pre-computed embeddings
class EmbeddingDataset(torch.utils.data.Dataset):
    """Simple dataset that holds embeddings on CPU."""
    def __init__(self, vjepa_embs, clip_embs):
        self.vjepa = vjepa_embs  # CPU tensors
        self.clip = clip_embs    # CPU tensors
        
    def __len__(self):
        return len(self.vjepa)
    
    def __getitem__(self, idx):
        return self.vjepa[idx], self.clip[idx]

def collate_cpu(batch):
    """Stack batch items without moving to GPU."""
    vjepa = torch.stack([b[0] for b in batch])
    clip = torch.stack([b[1] for b in batch])
    return vjepa, clip  # Both on CPU

train_dataset = EmbeddingDataset(X_train, Y_train)
test_dataset = EmbeddingDataset(X_test, Y_test)

train_loader = DataLoader(
    train_dataset, 
    batch_size=CONFIG['batch_size'], 
    shuffle=True, 
    collate_fn=collate_cpu,
    num_workers=0,  # Keep simple for CPU data
    pin_memory=True  # Faster CPU->GPU transfer
)
test_loader = DataLoader(
    test_dataset, 
    batch_size=CONFIG['batch_size'], 
    shuffle=False, 
    collate_fn=collate_cpu,
    num_workers=0,
    pin_memory=True
)

print(f"✅ DataLoaders ready:")
print(f"   Train: {len(train_loader)} batches ({len(train_dataset)} samples)")
print(f"   Test:  {len(test_loader)} batches ({len(test_dataset)} samples)")
print(f"   Batch size: {CONFIG['batch_size']}")
print(f"   Data stored on CPU, transferred to GPU per-batch")

In [ ]:
# =============================================================================
# TRAINING LOOP (Memory Optimized)
# =============================================================================
# - Batch-wise CPU->GPU transfer (embeddings stay on CPU until needed)
# - Automatic Mixed Precision (AMP) for faster training & less memory
# - Gradient accumulation for effective larger batch sizes
# - Explicit memory cleanup after each batch
# =============================================================================

def info_nce_loss(translated, targets, temperature=0.07):
    """InfoNCE contrastive loss."""
    translated = F.normalize(translated, dim=-1)
    targets = F.normalize(targets, dim=-1)
    logits = torch.matmul(translated, targets.T) / temperature
    labels = torch.arange(len(logits), device=logits.device)
    return (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)) / 2

# Setup
optimizer = torch.optim.AdamW(
    translator.parameters(), 
    lr=CONFIG['learning_rate'], 
    weight_decay=CONFIG['weight_decay']
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['num_epochs'])
scaler = torch.cuda.amp.GradScaler(enabled=CONFIG['use_amp'])
accumulation_steps = CONFIG['gradient_accumulation_steps']

history = {'train_loss': [], 'val_loss': [], 'learning_rate': [], 'gpu_memory': []}
best_val_loss = float('inf')

print(f"Training Configuration:")
print(f"  Epochs: {CONFIG['num_epochs']}")
print(f"  Batch size: {CONFIG['batch_size']} (effective: {CONFIG['batch_size'] * accumulation_steps})")
print(f"  Learning rate: {CONFIG['learning_rate']}")
print(f"  AMP: {CONFIG['use_amp']}")
print(f"  Gradient accumulation: {accumulation_steps} steps")
print(f"  Initial GPU memory: {get_gpu_memory_mb():.0f} MB\n")

for epoch in range(CONFIG['num_epochs']):
    # =========== TRAIN ===========
    translator.train()
    train_losses = []
    optimizer.zero_grad()
    
    pbar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch+1}/{CONFIG['num_epochs']} [Train]")
    for batch_idx, (vjepa_batch, clip_batch) in pbar:
        # Move batch to GPU
        vjepa_batch = vjepa_batch.to(device, non_blocking=True)
        clip_batch = clip_batch.to(device, non_blocking=True)
        
        # Forward pass with AMP
        with torch.cuda.amp.autocast(enabled=CONFIG['use_amp']):
            translated = translator(vjepa_batch)
            loss = info_nce_loss(translated, clip_batch, CONFIG['temperature'])
            loss = loss / accumulation_steps  # Scale for accumulation
        
        # Backward pass with gradient scaling
        scaler.scale(loss).backward()
        train_losses.append(loss.item() * accumulation_steps)
        
        # Gradient step every accumulation_steps
        if (batch_idx + 1) % accumulation_steps == 0 or (batch_idx + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(translator.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        
        # Free GPU memory
        del vjepa_batch, clip_batch, translated, loss
        
        pbar.set_postfix({'loss': f'{np.mean(train_losses[-10:]):.4f}'})
    
    # =========== VALIDATE ===========
    translator.eval()
    val_losses = []
    
    with torch.no_grad():
        for vjepa_batch, clip_batch in tqdm(test_loader, desc=f"Epoch {epoch+1}/{CONFIG['num_epochs']} [Val]", leave=False):
            vjepa_batch = vjepa_batch.to(device, non_blocking=True)
            clip_batch = clip_batch.to(device, non_blocking=True)
            
            with torch.cuda.amp.autocast(enabled=CONFIG['use_amp']):
                translated = translator(vjepa_batch)
                loss = info_nce_loss(translated, clip_batch, CONFIG['temperature'])
            
            val_losses.append(loss.item())
            del vjepa_batch, clip_batch, translated, loss
    
    # =========== EPOCH END ===========
    scheduler.step()
    clear_gpu_memory()
    
    avg_train = np.mean(train_losses)
    avg_val = np.mean(val_losses)
    gpu_mem = get_gpu_memory_mb()
    
    history['train_loss'].append(avg_train)
    history['val_loss'].append(avg_val)
    history['learning_rate'].append(scheduler.get_last_lr()[0])
    history['gpu_memory'].append(gpu_mem)
    
    # Checkpointing
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': translator.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': avg_val,
            'config': CONFIG
        }, 'best_translator_v2.pt')
        print(f"Epoch {epoch+1}: train={avg_train:.4f}, val={avg_val:.4f}, GPU={gpu_mem:.0f}MB ★ (saved)")
    else:
        print(f"Epoch {epoch+1}: train={avg_train:.4f}, val={avg_val:.4f}, GPU={gpu_mem:.0f}MB")

print(f"\n✅ Training complete!")
print(f"   Best validation loss: {best_val_loss:.4f}")
print(f"   Model saved: best_translator_v2.pt")

In [ ]:
# =============================================================================
# TRAINING VISUALIZATION
# =============================================================================

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["train_loss"], label="Train")
axes[0].plot(history["val_loss"], label="Val")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].plot(history["learning_rate"], color="green")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("LR"); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()

In [ ]:
# =============================================================================
# EVALUATION (Memory Optimized)
# =============================================================================

# Load best model
checkpoint = torch.load('best_translator_v2.pt', weights_only=False)
translator.load_state_dict(checkpoint['model_state_dict'])
translator.eval()
print(f"Loaded best model from epoch {checkpoint['epoch']} (val_loss: {checkpoint['val_loss']:.4f})")

# Evaluate on test set
all_translated = []
all_targets = []

with torch.no_grad():
    for vjepa_batch, clip_batch in tqdm(test_loader, desc="Evaluating"):
        vjepa_batch = vjepa_batch.to(device, non_blocking=True)
        
        with torch.cuda.amp.autocast(enabled=CONFIG['use_amp']):
            translated = translator(vjepa_batch)
        
        all_translated.append(translated.float().cpu())
        all_targets.append(clip_batch)  # Already on CPU
        
        del vjepa_batch, translated
    
    clear_gpu_memory()

all_translated = torch.cat(all_translated)
all_targets = torch.cat(all_targets)

# Compute similarity matrix
similarity = torch.matmul(F.normalize(all_translated, dim=-1), 
                         F.normalize(all_targets, dim=-1).T)

# Metrics
diagonal_sim = similarity.diag().mean().item()
labels = torch.arange(len(similarity))
top1 = (similarity.argmax(dim=1) == labels).float().mean().item()
top5 = sum(labels[i] in similarity[i].topk(5).indices for i in range(len(labels))) / len(labels)
top10 = sum(labels[i] in similarity[i].topk(10).indices for i in range(len(labels))) / len(labels)

print(f"\n📊 Evaluation Results:")
print(f"   Mean diagonal similarity: {diagonal_sim:.4f}")
print(f"   Top-1 retrieval accuracy: {top1*100:.1f}%")
print(f"   Top-5 retrieval accuracy: {top5*100:.1f}%")
print(f"   Top-10 retrieval accuracy: {top10*100:.1f}%")

In [ ]:
# =============================================================================
# SAVE MODEL
# =============================================================================

import json

torch.save({"model_state_dict": translator.state_dict(), "config": CONFIG, "history": history}, "vjepa_to_clip_translator.pt")
with open("translator_config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)

print("Saved: vjepa_to_clip_translator.pt, translator_config.json")

In [ ]:
# ============================================================================
# VIDEO DOWNLOAD UTILITIES
# ============================================================================

import subprocess
import requests

VIDEO_DIR = Path(CONFIG['video_dir'])
VIDEO_DIR.mkdir(exist_ok=True)
VIDEO_EXTENSIONS = ['.mp4', '.avi', '.mov', '.mkv', '.webm']


def count_videos():
    """Count existing videos"""
    return len([f for f in VIDEO_DIR.iterdir() if f.suffix.lower() in VIDEO_EXTENSIONS])


def list_videos():
    """List all videos"""
    return sorted([f for f in VIDEO_DIR.iterdir() if f.suffix.lower() in VIDEO_EXTENSIONS])


def download_file(url, output_path, timeout=120):
    """Download a file with error handling"""
    try:
        response = requests.get(url, timeout=timeout, stream=True)
        if response.status_code == 200:
            with open(output_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            if output_path.exists() and output_path.stat().st_size > 10000:
                return True
        output_path.unlink(missing_ok=True)
    except Exception:
        output_path.unlink(missing_ok=True)
    return False


SAMPLE_VIDEOS = [
    ("blender_bunny", "https://download.blender.org/peach/trailer/trailer_480p.mov"),
    ("blender_sintel", "https://download.blender.org/durian/trailer/sintel_trailer-480p.mp4"),
    ("test_bars", "https://www.w3schools.com/html/mov_bbb.mp4"),
]


def download_sample_videos():
    """Download sample videos from public sources"""
    print("\n📥 Downloading sample videos...")
    for name, url in SAMPLE_VIDEOS:
        ext = '.mp4' if '.mp4' in url else '.mov'
        output_path = VIDEO_DIR / f"sample_{name}{ext}"
        if output_path.exists():
            print(f"  ✓ {name} (exists)")
            continue
        print(f"  ⬇ {name}...", end=" ")
        if download_file(url, output_path):
            print("✓")
        else:
            print("✗")


def generate_synthetic_videos(n_videos=20, duration=3, fps=24, resolution=(256, 256)):
    """Generate synthetic test videos with moving shapes"""
    print(f"\n📥 Generating {n_videos} synthetic videos...")
    try:
        import cv2
    except ImportError:
        subprocess.run(["pip", "install", "opencv-python-headless"], capture_output=True)
        import cv2
    
    for i in tqdm(range(n_videos), desc="  Generating"):
        output_path = VIDEO_DIR / f"synthetic_{i:03d}.mp4"
        if output_path.exists():
            continue
        
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(str(output_path), fourcc, fps, resolution)
        h, w = resolution
        
        for frame_idx in range(duration * fps):
            t = frame_idx / (duration * fps)
            frame = np.zeros((h, w, 3), dtype=np.uint8)
            
            pattern = i % 4
            if pattern == 0:  # Moving circle
                cx = int(w * (0.2 + 0.6 * t))
                cy = int(h * (0.5 + 0.3 * np.sin(t * 4 * np.pi)))
                cv2.circle(frame, (cx, cy), 30, (int(255*t), int(255*(1-t)), 128), -1)
            elif pattern == 1:  # Expanding rectangle
                size = int(20 + 80 * t)
                cv2.rectangle(frame, (w//2-size, h//2-size), (w//2+size, h//2+size), (0, 255, 255), -1)
            elif pattern == 2:  # Bouncing ball
                cx = int(w * (0.2 + 0.6 * abs(np.sin(t * 3 * np.pi))))
                cy = int(h * (0.2 + 0.6 * abs(np.cos(t * 2 * np.pi))))
                cv2.circle(frame, (cx, cy), 25, (255, 100, 100), -1)
            else:  # Rotating line
                angle = t * 4 * np.pi
                length = min(w, h) // 3
                x1 = int(w/2 + length * np.cos(angle))
                y1 = int(h/2 + length * np.sin(angle))
                x2 = int(w/2 - length * np.cos(angle))
                y2 = int(h/2 - length * np.sin(angle))
                cv2.line(frame, (x1, y1), (x2, y2), (100, 200, 255), 8)
            
            out.write(frame)
        out.release()


def download_videos(target_count=50):
    """Download/generate videos until target count reached"""
    print(f"📂 Current videos: {count_videos()}")
    if count_videos() >= target_count:
        print(f"✅ Already have {count_videos()} videos")
        return
    
    download_sample_videos()
    if count_videos() < target_count:
        generate_synthetic_videos(n_videos=target_count - count_videos())
    
    print(f"\n✅ Total videos: {count_videos()}")


print(f"📂 Video directory: {VIDEO_DIR.absolute()}")
print(f"📊 Current count: {count_videos()} videos")

In [ ]:
# ============================================================================
# DOWNLOAD VIDEOS (run this cell to get training data)
# ============================================================================

TARGET_VIDEO_COUNT = 100
download_videos(target_count=TARGET_VIDEO_COUNT)

video_files = list_videos()
print(f"\n✅ Ready: {len(video_files)} videos")

In [ ]:
# ============================================================================
# LOAD MODELS
# ============================================================================

from transformers import AutoModel, AutoVideoProcessor, CLIPModel, CLIPProcessor

# Load V-JEPA2
print("Loading V-JEPA2...")
vjepa_model = AutoModel.from_pretrained(
    CONFIG['vjepa_model'], 
    torch_dtype=torch.float16,
    trust_remote_code=True
)
vjepa_model = vjepa_model.to(device).eval()
vjepa_processor = AutoVideoProcessor.from_pretrained(CONFIG['vjepa_model'])
vjepa_dim = vjepa_model.config.hidden_size
print(f"V-JEPA2: {sum(p.numel() for p in vjepa_model.parameters())/1e6:.1f}M params, dim={vjepa_dim}")

# Load CLIP
print("\nLoading CLIP...")
clip_model = CLIPModel.from_pretrained(CONFIG['clip_model'])
clip_model = clip_model.to(device).eval()
clip_processor = CLIPProcessor.from_pretrained(CONFIG['clip_model'])
clip_dim = clip_model.config.projection_dim
print(f"CLIP: {sum(p.numel() for p in clip_model.parameters())/1e6:.1f}M params, dim={clip_dim}")

In [ ]:
# ============================================================================
# VIDEO UTILITIES - WINDOWED CONSECUTIVE FRAME EXTRACTION
# ============================================================================
# V-JEPA uses 3D space-time patches - temporal coherence MATTERS!
# We extract CONSECUTIVE frames, not randomly sampled ones.
# ============================================================================

import av
from PIL import Image


def get_video_frame_count(video_path):
    """Get total frame count of a video"""
    try:
        container = av.open(str(video_path))
        stream = container.streams.video[0]
        count = stream.frames or sum(1 for _ in container.decode(video=0))
        container.close()
        return count
    except:
        return 0


def extract_consecutive_frames(video_path, start_frame, num_frames):
    """
    Extract consecutive frames starting at start_frame.
    Pads with last frame if needed.
    """
    container = av.open(str(video_path))
    frames = []
    
    for i, frame in enumerate(container.decode(video=0)):
        if i >= start_frame:
            frames.append(frame.to_ndarray(format='rgb24'))
        if len(frames) >= num_frames:
            break
    container.close()
    
    while len(frames) < num_frames:
        frames.append(frames[-1] if frames else np.zeros((224, 224, 3), dtype=np.uint8))
    
    return frames[:num_frames]


def get_window_starts(total_frames, window_size, stride, max_windows=None):
    """
    Calculate window start positions for a video.
    
    Example: total=200, window=64, stride=32 → [0, 32, 64, 96, 128]
    """
    if total_frames < window_size:
        return [0]
    
    starts = []
    pos = 0
    while pos + window_size <= total_frames:
        starts.append(pos)
        pos += stride
    
    if max_windows and len(starts) > max_windows:
        indices = np.linspace(0, len(starts)-1, max_windows, dtype=int)
        starts = [starts[i] for i in indices]
    
    return starts


print("✓ Video utilities loaded")
print(f"  Window: {CONFIG['num_frames']} frames, stride={CONFIG['window_stride']}")

In [ ]:
# ============================================================================
# BUILD WINDOW INDEX & EXTRACT EMBEDDINGS
# ============================================================================
# Scan videos, build window index, extract V-JEPA and CLIP embeddings.
# Each window is one training sample.
# ============================================================================

print("Building window index...")
window_index = []  # (video_path, start_frame)

for vpath in tqdm(video_files, desc="Scanning videos"):
    total_frames = get_video_frame_count(vpath)
    if total_frames < CONFIG['min_video_frames']:
        continue
    
    starts = get_window_starts(
        total_frames,
        CONFIG['num_frames'],
        CONFIG['window_stride'],
        CONFIG['max_windows_per_video']
    )
    for start in starts:
        window_index.append((vpath, start))

print(f"\n✅ {len(window_index)} windows from {len(set(w[0] for w in window_index))} videos")

# ============================================================================
# EXTRACT EMBEDDINGS
# ============================================================================

print("\nExtracting embeddings (this may take a while)...")
vjepa_embeddings = []
clip_embeddings = []
valid_windows = []

for vpath, start_frame in tqdm(window_index, desc="Extracting"):
    try:
        # Extract consecutive frames
        frames = extract_consecutive_frames(vpath, start_frame, CONFIG['num_frames'])
        
        # V-JEPA embedding
        video_array = np.stack(frames, axis=0)
        inputs = vjepa_processor(video_array, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = vjepa_model(**inputs)
            vjepa_emb = outputs.last_hidden_state.squeeze(0).mean(dim=0).cpu().float()
        
        # CLIP embedding (average of sampled frames)
        indices = np.linspace(0, len(frames)-1, CONFIG['clip_sample_frames'], dtype=int)
        clip_embs = []
        for idx in indices:
            img = Image.fromarray(frames[idx])
            clip_inputs = clip_processor(images=img, return_tensors="pt")
            clip_inputs = {k: v.to(device) for k, v in clip_inputs.items()}
            with torch.no_grad():
                emb = clip_model.get_image_features(**clip_inputs)
            clip_embs.append(emb.squeeze())
        clip_emb = torch.stack(clip_embs).mean(dim=0).cpu().float()
        
        vjepa_embeddings.append(vjepa_emb)
        clip_embeddings.append(clip_emb)
        valid_windows.append((vpath, start_frame))
        
    except Exception as e:
        continue

vjepa_embeddings = torch.stack(vjepa_embeddings)
clip_embeddings = torch.stack(clip_embeddings)

print(f"\n✅ Extracted embeddings:")
print(f"   V-JEPA: {vjepa_embeddings.shape}")
print(f"   CLIP:   {clip_embeddings.shape}")

In [ ]:
# ============================================================================
# TRAIN/TEST SPLIT (VIDEO LEVEL)
# ============================================================================
# Split by VIDEO to prevent data leakage!
# ============================================================================

unique_videos = list(set([w[0] for w in valid_windows]))
train_videos, test_videos = train_test_split(
    unique_videos, 
    test_size=CONFIG['test_split'], 
    random_state=CONFIG['random_seed']
)
train_videos_set = set(train_videos)
test_videos_set = set(test_videos)

train_idx = [i for i, w in enumerate(valid_windows) if w[0] in train_videos_set]
test_idx = [i for i, w in enumerate(valid_windows) if w[0] in test_videos_set]

X_train = vjepa_embeddings[train_idx].to(device)
Y_train = clip_embeddings[train_idx].to(device)
X_test = vjepa_embeddings[test_idx].to(device)
Y_test = clip_embeddings[test_idx].to(device)

print(f"Video-level split:")
print(f"  Train: {len(train_videos)} videos → {len(train_idx)} windows")
print(f"  Test:  {len(test_videos)} videos → {len(test_idx)} windows")

In [ ]:
# ============================================================================
# TRANSLATOR NETWORK
# ============================================================================

class ResidualBlock(nn.Module):
    def __init__(self, dim, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.ff = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * 4, dim),
            nn.Dropout(dropout)
        )
    
    def forward(self, x):
        return x + self.ff(self.norm(x))


class Translator(nn.Module):
    """V-JEPA → CLIP translator with residual blocks"""
    def __init__(self, input_dim, output_dim, hidden_dim=512, num_blocks=4, dropout=0.1):
        super().__init__()
        
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        
        self.blocks = nn.ModuleList([
            ResidualBlock(hidden_dim, dropout) for _ in range(num_blocks)
        ])
        
        self.output_proj = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, output_dim)
        )
        
        self.apply(self._init_weights)
    
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
    
    def forward(self, x):
        x = self.input_proj(x)
        for block in self.blocks:
            x = block(x)
        x = self.output_proj(x)
        return F.normalize(x, dim=-1)


# Adjust for dataset size
if len(X_train) < 200:
    CONFIG['translator_hidden_dim'] = 256
    CONFIG['translator_num_blocks'] = 2
    CONFIG['translator_dropout'] = 0.2

translator = Translator(
    input_dim=vjepa_dim,
    output_dim=clip_dim,
    hidden_dim=CONFIG['translator_hidden_dim'],
    num_blocks=CONFIG['translator_num_blocks'],
    dropout=CONFIG['translator_dropout']
).to(device)

print(f"Translator: {sum(p.numel() for p in translator.parameters())/1e6:.2f}M params")

In [ ]:
# ============================================================================
# LOSS FUNCTIONS
# ============================================================================

def contrastive_loss(pred, target, temperature=0.07):
    """Symmetric InfoNCE contrastive loss"""
    pred_norm = F.normalize(pred, dim=1)
    target_norm = F.normalize(target, dim=1)
    
    logits = pred_norm @ target_norm.T / temperature
    labels = torch.arange(len(pred), device=pred.device)
    
    loss = (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)) / 2
    
    with torch.no_grad():
        acc = (logits.argmax(dim=1) == labels).float().mean()
    
    return loss, acc


def cosine_loss(pred, target):
    """1 - cosine similarity"""
    pred_norm = F.normalize(pred, dim=1)
    target_norm = F.normalize(target, dim=1)
    return 1 - (pred_norm * target_norm).sum(dim=1).mean()


print(f"✓ Loss functions defined")
print(f"  Contrastive weight: {CONFIG['contrastive_weight']}")
print(f"  Cosine weight: {CONFIG['cosine_weight']}")

In [ ]:
# ============================================================================
# TRAINING
# ============================================================================

# Adjust batch size for small datasets
batch_size = min(CONFIG['batch_size'], len(X_train))

optimizer = torch.optim.AdamW(
    translator.parameters(),
    lr=CONFIG['lr'],
    weight_decay=CONFIG['weight_decay'],
    betas=(0.9, 0.98)
)

def lr_lambda(epoch):
    if epoch < CONFIG['warmup_epochs']:
        return epoch / CONFIG['warmup_epochs']
    progress = (epoch - CONFIG['warmup_epochs']) / (CONFIG['epochs'] - CONFIG['warmup_epochs'])
    return 0.5 * (1 + np.cos(np.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# Training loop
history = {'train_loss': [], 'test_loss': [], 'train_acc': [], 'test_acc': [], 'lr': []}
best_test_loss = float('inf')
best_epoch = 0
patience_counter = 0
best_state = None

print(f"Training for up to {CONFIG['epochs']} epochs...")
print(f"Early stopping patience: {CONFIG['patience']}")
print("-" * 70)

for epoch in range(CONFIG['epochs']):
    translator.train()
    
    perm = torch.randperm(len(X_train))
    epoch_loss, epoch_acc, n_batches = 0, 0, 0
    
    for i in range(0, len(X_train), batch_size):
        batch_idx = perm[i:i+batch_size]
        x_batch = X_train[batch_idx]
        y_batch = Y_train[batch_idx]
        
        optimizer.zero_grad()
        pred = translator(x_batch)
        
        loss_contr, acc = contrastive_loss(pred, y_batch, CONFIG['temperature'])
        loss_cos = cosine_loss(pred, y_batch)
        loss = CONFIG['contrastive_weight'] * loss_contr + CONFIG['cosine_weight'] * loss_cos
        
        loss.backward()
        if CONFIG['grad_clip'] > 0:
            torch.nn.utils.clip_grad_norm_(translator.parameters(), CONFIG['grad_clip'])
        optimizer.step()
        
        epoch_loss += loss.item()
        epoch_acc += acc.item()
        n_batches += 1
    
    scheduler.step()
    
    # Evaluation
    translator.eval()
    with torch.no_grad():
        pred_test = translator(X_test)
        test_contr, test_acc = contrastive_loss(pred_test, Y_test, CONFIG['temperature'])
        test_cos = cosine_loss(pred_test, Y_test)
        test_loss = CONFIG['contrastive_weight'] * test_contr + CONFIG['cosine_weight'] * test_cos
    
    history['train_loss'].append(epoch_loss / n_batches)
    history['test_loss'].append(test_loss.item())
    history['train_acc'].append(epoch_acc / n_batches)
    history['test_acc'].append(test_acc.item())
    history['lr'].append(optimizer.param_groups[0]['lr'])
    
    # Early stopping
    if test_loss < best_test_loss:
        best_test_loss = test_loss.item()
        best_epoch = epoch
        patience_counter = 0
        best_state = {k: v.cpu().clone() for k, v in translator.state_dict().items()}
    else:
        patience_counter += 1
    
    if (epoch + 1) % 50 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:4d}: loss={history['train_loss'][-1]:.4f}/{test_loss:.4f}, "
              f"acc={history['train_acc'][-1]*100:.1f}%/{test_acc*100:.1f}%")
    
    if patience_counter >= CONFIG['patience']:
        print(f"\n⏹️ Early stopping at epoch {epoch+1} (best: {best_epoch+1})")
        break

translator.load_state_dict({k: v.to(device) for k, v in best_state.items()})
print("-" * 70)
print(f"✅ Training complete! Best: {best_test_loss:.4f} at epoch {best_epoch+1}")

In [ ]:
# ============================================================================
# PLOT TRAINING CURVES
# ============================================================================

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ax1 = axes[0]
ax1.semilogy(history['train_loss'], label='Train', alpha=0.8)
ax1.semilogy(history['test_loss'], label='Test', alpha=0.8)
ax1.axvline(best_epoch, color='g', linestyle='--', alpha=0.5, label=f'Best ({best_epoch+1})')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2 = axes[1]
ax2.plot([x*100 for x in history['train_acc']], label='Train', alpha=0.8)
ax2.plot([x*100 for x in history['test_acc']], label='Test', alpha=0.8)
ax2.axvline(best_epoch, color='g', linestyle='--', alpha=0.5)
ax2.axhline(y=100/len(X_test), color='r', linestyle=':', alpha=0.5, label='Random')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Retrieval Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

ax3 = axes[2]
ax3.plot(history['lr'])
ax3.set_xlabel('Epoch')
ax3.set_ylabel('Learning Rate')
ax3.set_title('LR Schedule')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nBest epoch: {best_epoch+1}")
print(f"Best test accuracy: {history['test_acc'][best_epoch]*100:.1f}%")

In [ ]:
# ============================================================================
# SAVE MODEL
# ============================================================================

save_path = Path('translator_vjepa_to_clip_v2.pt')

save_data = {
    'model_state_dict': translator.state_dict(),
    'vjepa_dim': vjepa_dim,
    'clip_dim': clip_dim,
    'config': CONFIG,
    'best_epoch': best_epoch,
    'best_test_loss': best_test_loss,
    'best_test_acc': history['test_acc'][best_epoch],
    'history': history,
    'n_train_windows': len(train_idx),
    'n_test_windows': len(test_idx),
    'n_train_videos': len(train_videos),
    'n_test_videos': len(test_videos),
}

torch.save(save_data, save_path)
print(f"✅ Saved to {save_path}")
print(f"   Best accuracy: {history['test_acc'][best_epoch]*100:.1f}%")
print(f"   Windows: {len(train_idx)} train, {len(test_idx)} test")